# Ejercicios

## [01] Leer un CSV del proveedor europeo: separador y decimales

El proveedor "EuroFoods" te envía un CSV con separador punto y coma (;) y decimales con coma. 

Léelo correctamente y calcula el total de importes.


Resultado esperado

Columnas detectadas: ['producto', 'precio', 'cantidad']

- Jamón ibérico        → 459.00€
- Queso manchego       → 312.50€
- Aceite de oliva      → 449.50€
- Vino Rioja           → 450.00€

TOTAL: 1671.00€
▲ colapsar

In [32]:
import csv, io

In [33]:
# El CSV que manda el proveedor europeo. Fíjate en dos cosas: cómo separa
# las columnas y cómo escribe los decimales.
csv_texto = """producto;precio;cantidad
Jamón ibérico;45,90;10
Queso manchego;12,50;25
Aceite de oliva;8,99;50
Vino Rioja;15,00;30
"""

In [34]:
SEPARADOR = ";"      # <- ¿de verdad este CSV separa por comas?

reader = csv.DictReader(io.StringIO(csv_texto), delimiter=SEPARADOR)

In [35]:
# No toques: enseña qué columnas ha entendido el lector con ese separador
print("Columnas detectadas:", reader.fieldnames)
print("")

Columnas detectadas: ['producto', 'precio', 'cantidad']



In [36]:
reader = list(reader)

In [37]:
for fila in reader:
    print(fila)

{'producto': 'Jamón ibérico', 'precio': '45,90', 'cantidad': '10'}
{'producto': 'Queso manchego', 'precio': '12,50', 'cantidad': '25'}
{'producto': 'Aceite de oliva', 'precio': '8,99', 'cantidad': '50'}
{'producto': 'Vino Rioja', 'precio': '15,00', 'cantidad': '30'}


In [40]:
total = 0
registros = []
for fila in reader:
    precio = float(fila['precio'].replace(",", "."))     # <- pasa fila["precio"] a número. Ojo: viene como "45,90"
    cantidad = int(fila['cantidad'])     # <- pasa fila["cantidad"] a entero
    subtotal = precio * cantidad
    total += subtotal
    print(precio)
    registros.append({"producto": fila.get("producto", "???"), "subtotal": subtotal})

print(registros)

45.9
12.5
8.99
15.0
[{'producto': 'Jamón ibérico', 'subtotal': 459.0}, {'producto': 'Queso manchego', 'subtotal': 312.5}, {'producto': 'Aceite de oliva', 'subtotal': 449.5}, {'producto': 'Vino Rioja', 'subtotal': 450.0}]


In [41]:
# No toques
for r in registros:
    print(f"  {r['producto']:20} → {r['subtotal']:.2f}€")
print(f"\nTOTAL: {total:.2f}€")

  Jamón ibérico        → 459.00€
  Queso manchego       → 312.50€
  Aceite de oliva      → 449.50€
  Vino Rioja           → 450.00€

TOTAL: 1671.00€


## [02] Convertir JSON anidado a CSV plano



Aplana productos con specs anidadas y escríbelos como CSV. Ojo: un 0 SÍ es un dato, no lo confundas con "no hay dato".



Resultado esperado

- nombre,precio,ram_gb,disco_gb,tag_principal
- MacBook Pro,2499,16,512,portátil
- iPad Air,699,8,256,tablet
- AirPods Pro,279,,,audio
- Funda iPad,59,,0,

In [42]:
productos_json = [
    {"nombre": "MacBook Pro", "precio": 2499, "specs": {"ram": 16, "disco": 512}, "tags": ["portátil", "pro"]},
    {"nombre": "iPad Air", "precio": 699, "specs": {"ram": 8, "disco": 256}, "tags": ["tablet"]},
    {"nombre": "AirPods Pro", "precio": 279, "specs": {"ram": None, "disco": None}, "tags": ["audio", "wireless"]},
    {"nombre": "Funda iPad", "precio": 59, "specs": {"ram": None, "disco": 0}, "tags": []},
]

In [65]:
# Aplana cada producto y escríbelo como fila del CSV.
# Ojo: el CSV no tiene "null". Donde no haya dato, escribe "".
# Y más ojo aún: un 0 SÍ es un dato. No lo confundas con "no hay dato".
# Pista: "is not None" distingue los dos; "if x" los confunde.
with open("productos.csv", "w", newline="", encoding="utf-8") as f:
    campos = ["nombre", "precio", "ram_gb", "disco_gb", "tag_principal"]
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()

    for prod in productos_json:
        fila = {
            "nombre": prod['nombre'],
            "precio": prod['precio'],
            "ram_gb": prod['specs']['ram'] if prod['specs']['ram'] is not None else None,
            "disco_gb": prod['specs']['disco'] if prod['specs']['disco'] is not None else None,
            "tag_principal": prod['tags'][0] if prod['tags'] else None,
        }
        writer.writerow(fila)


In [66]:
# No toques: vuelve a abrir el fichero para ver qué has escrito de verdad
with open("productos.csv", "r", encoding="utf-8") as f:
    print(f.read(), end="")

nombre,precio,ram_gb,disco_gb,tag_principal
MacBook Pro,2499,16,512,portátil
iPad Air,699,8,256,tablet
AirPods Pro,279,,,audio
Funda iPad,59,,0,


## [03] Procesar JSONL línea a línea



Un JSONL de eventos de e-commerce. Procesa línea a línea y calcula estadísticas. Ojo: una compra puede no traer importe.

Resultado esperado

- Total eventos: 10
- Compras: 6
- Facturación: 416.49€
- Usuarios únicos: 5
- Ticket medio: 69.42€   (sobre las 6 compras)
- Ticket medio: 83.30€   (sólo las 5 con importe)

In [68]:
import json

# Un JSONL: un objeto JSON por línea. En producción serían millones.
jsonl_data = """{"tipo":"compra","importe":50.0,"usuario":"ana"}
{"tipo":"login","usuario":"pedro"}
{"tipo":"compra","importe":120.5,"usuario":"lucia"}
{"tipo":"compra","importe":30.0,"usuario":"ana"}
{"tipo":"logout","usuario":"pedro"}
{"tipo":"compra","importe":200.0,"usuario":"carlos"}
{"tipo":"login","usuario":"ana"}
{"tipo":"compra","importe":15.99,"usuario":"lucia"}
{"tipo":"compra","usuario":"marta"}
{"tipo":"error","codigo":500}
"""


In [77]:
total_eventos = 0
total_compras = 0
compras_con_importe = 0
suma_importes = 0
usuarios_unicos = set()    # un conjunto: no admite repetidos

for linea in jsonl_data.strip().split("\n"):
    evento: dict = json.loads(linea)                  # <- convierte la línea en diccionario. Pista: json.loads

    total_eventos += 1

    if evento.get("tipo") == "compra":
        total_compras += 1
        importe = evento.get('importe', 0) 
        suma_importes += importe # <- suma aquí el importe, sin que reviente si el evento no lo trae

        if importe != 0:
            compras_con_importe +=1 # <- cuenta aparte cuántas compras SÍ traen importe

    if evento.get('usuario'):
        usuarios_unicos.add(evento['usuario'])    # <- si el evento tiene "usuario", guárdalo. Pista: usuarios_unicos.add(...)

usuarios_unicos

{'ana', 'carlos', 'lucia', 'marta', 'pedro'}

In [78]:
# No toques
print(f"Total eventos: {total_eventos}")
print(f"Compras: {total_compras}")
print(f"Facturación: {suma_importes:.2f}€")
print(f"Usuarios únicos: {len(usuarios_unicos)}")
print(f"Ticket medio: {suma_importes / total_compras:.2f}€   (sobre las {total_compras} compras)")
print(f"Ticket medio: {suma_importes / compras_con_importe:.2f}€   (sólo las {compras_con_importe} con importe)")

Total eventos: 10
Compras: 6
Facturación: 416.49€
Usuarios únicos: 5
Ticket medio: 69.42€   (sobre las 6 compras)
Ticket medio: 83.30€   (sólo las 5 con importe)


## [04] Validador de CSV: detectar filas rotas


- Un CSV tiene filas con problemas. La primera comprobación (número de columnas) ya está hecha.
- Añade las otras dos: nombre vacío y edad no numérica.
- La comprobación compara el texto EXACTO de cada error, así que copia los dos mensajes tal cual, con su tilde y sus comillas: "nombre vacío" y f"edad no numérica: '{fila[3]}'". - Si tu lógica es correcta pero falla, casi siempre es una tilde o una comilla que no coincide, no tu código.

Resultado esperado

- Filas válidas: 2
- Filas con error: 4
  - Línea 3: ['columnas: esperadas 4, tiene 3']
  - Línea 4: ['columnas: esperadas 4, tiene 5']

In [79]:
import io

csv_sucio = """id,nombre,email,edad
1,Ana,ana@mail.com,28
2,Pedro,pedro@mail.com
3,Lucía,lucia@mail.com,35,extra_campo
4,Carlos,carlos@mail.com,40
5,,sara@mail.com,22
6,Jorge,jorge@mail.com,treinta
"""


In [93]:
from os import error


columnas_esperadas = 4
filas_ok = []
filas_error = []

# io.StringIO hace que un texto se comporte como si fuera un fichero
reader = csv.reader(io.StringIO(csv_sucio.strip()))
header = next(reader)          # next() consume la primera línea: la cabecera

for i, fila in enumerate(reader, start=2):     # el header es la línea 1
    errores = []

    # Tres cosas que comprobar. La primera ya está hecha:
    if len(fila) != columnas_esperadas:
        errores.append(f"columnas: esperadas {columnas_esperadas}, tiene {len(fila)}")

    if len(fila) > 1 and fila[1].strip() == "":
        errores.append("nombre vacío") # <- ¿el nombre (columna 1) está vacío?  -> "nombre vacío"

    if len(fila) >= columnas_esperadas: # <- ¿la edad (columna 3) es un número?  -> f"edad no numérica: '{fila[3]}'"
    #    Ojo: si la fila viene corta, fila[3] no existe. Comprueba antes len(fila).
        try:
            edad = int(fila[3])
        except:
            errores.append(f"edad no numérica: '{fila[3]}'")
        data = 0 

    if errores:
        filas_error.append({"linea": i, "errores": errores})
    else:
        filas_ok.append(i)


In [94]:
# No toques
print(f"Filas válidas: {len(filas_ok)}")
print(f"Filas con error: {len(filas_error)}")
for e in filas_error:
    print(f"  Línea {e['linea']}: {e['errores']}")

Filas válidas: 2
Filas con error: 4
  Línea 3: ['columnas: esperadas 4, tiene 3']
  Línea 4: ['columnas: esperadas 4, tiene 5']
  Línea 6: ['nombre vacío']
  Línea 7: ["edad no numérica: 'treinta'"]
